# Camada Silver
Nessa etapa trataremos as tabelas da etapa bronze, definindo tipos, lidando com nulos, eliminando dados invalidos e demais transformações necessárias

## VRA
Vamos inspecionar a tabela `mvp_engenharia_dados.bronze.bronze_vra` para determinar quais tratamentos serão necessàrios

In [0]:
%sql
DESCRIBE mvp_engenharia_dados.bronze.bronze_vra

### Duplicidade de dados

In [0]:
from pyspark.sql.functions import col

vra = spark.table("mvp_engenharia_dados.bronze.bronze_vra")

total_rows = vra.count()
distinct_rows = vra.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Total de linhas: {total_rows:,d}")
print(f"Linhas distintas: {distinct_rows:,d}")
print(f"Duplicatas: {duplicates:,d} ({duplicates / total_rows * 100:.2f}%)")

if duplicates > 0:
    print("\nTop 10 linhas duplicadas:")
    display(
        vra.groupBy(vra.columns)
          .count()
          .filter(col("count") > 1)
          .orderBy(col("count").desc())
          .limit(10)
    )

In [0]:
vra = vra.drop_duplicates()

### Valores Nulos ou em Branco

In [0]:
from pyspark.sql.functions import col, when, count, trim

total = vra.count()

null_row = vra.select([count(when(col(c).isNull(), c)).alias(c) for c in vra.columns]).collect()[0]
blank_row = vra.select([count(when((col(c).isNotNull()) & (trim(col(c)) == ""), c)).alias(c) for c in vra.columns]).collect()[0]

results = [(c, int(null_row[c]), int(blank_row[c])) for c in vra.columns]
display(spark.createDataFrame(results, ["column", "null_count", "blank_count"]))

> Nao ha valores em branco (''), somente nulos, o que simplifica o tratamento.

#### Tipo Linha nulos

In [0]:
%sql
select * from mvp_engenharia_dados.bronze.bronze_vra where `Código Tipo Linha` is null

> Voos com Código Tipo Linha em branco estao todos **CANCELADOS**.  
> Como de qualquer forma esses voos nao serao considerados, podemos ignora-los 

In [0]:
vra = vra.dropna(subset="Código Tipo Linha")

#### Partida Prevista nulos

In [0]:
display(vra.filter(col("Partida Prevista").isNull()).limit(25))

> Os voos sem Partida Prevista (Ou chegada Prevista), não devem ser contabilizados para o fim de pontualidade.  
> Eles distorceriam o índice, melhorando o resultado artificialmente

In [0]:
vra = vra.dropna(subset=["Partida Prevista", "Chegada Prevista"], how='any')

#### Partida Real e Chegada Real
Vamos verificar se as partidas realizadas obrigatoriamente tem chegadas realizadas (e vice versa)

In [0]:
%sql
SELECT
	*

FROM
	mvp_engenharia_dados.bronze.bronze_vra

WHERE
	(
		`Partida Real` IS NULL
		AND `Chegada Real` IS NOT NULL
	)
	OR (
		`Partida Real` IS NOT NULL
		AND `Chegada Real` IS NULL
	)

> As partidas e chegadas realizadas sempre possuem par, ou seja, nunca temos somente a partida ou a chegada registrada somente, sempre temos ambas.
> Podemos filtrar tranquilamente as viagens sem partidas real registrada

In [0]:
vra = vra.dropna(subset=["Partida Real", "Chegada Real"], how='any')

### Justificativa
O campo Justificativa deixou de ser obrigatorio a partir de 2020, por isso está totalmente nulo na base.  
Vamos eliminar essa coluna

In [0]:
vra = vra.drop("Justificativa")

### Codeshare
Não é um campo relevante para nossa analise, então tambem pode ser eliminado

In [0]:
vra = vra.drop("Codeshare")

### Nulos pos tratamento

In [0]:
total_anterior = spark.table("mvp_engenharia_dados.bronze.bronze_vra").count()
total = vra.count()

null_row = vra.select([count(when(col(c).isNull(), c)).alias(c) for c in vra.columns]).collect()[0]
blank_row = vra.select([count(when((col(c).isNotNull()) & (trim(col(c)) == ""), c)).alias(c) for c in vra.columns]).collect()[0]

results = [(c, int(null_row[c]), int(blank_row[c])) for c in vra.columns]
display(spark.createDataFrame(results, ["column", "null_count", "blank_count"]))

print(f"Total de linhas Antes: {total_anterior}")
print(f"Total de linhas Atual: {total}")
print(f"Percentual Perdido: {(total_anterior - total) / total_anterior * 100:.2f}%")


> Sem mais nenhum valor nulo nas colunas.
> Foram eliminados um total de 7% de linhas

## Filtros de regras

Como só vamos considerar voos nacionais e não cancelados nas nossas métricas, podemos eliminar os demais voos da tabela.

### Voos Cancelados

In [0]:
vra.filter(col("Situação Voo") == "CANCELADO").count()

> Os filtros de nulo anteriores acabaram já eliminando os Cancelados

In [0]:
display(vra.groupBy("Situação Voo").count())

> Ao meu ver não faz sentido manter essa coluna se a tabela só irá possuir voos realizados

In [0]:
vra = vra.drop("Situação Voo")

### Tipos de Linhas
Como só queremos as linhas domésticas de passageiros, usaremos o tipo da linha para determinar.

| Código | Descrição |
| --- | --- |
| **N** | **Doméstica Mista: transporte de passageiros ou mistos, com todos os aeródromos em território brasileiro** |
| C | Doméstica Cargueira: transporte exclusivo de carga e/ou mala postal, com todos os aeródromos em território brasileiro |
| I | Internacional Mista: transporte de passageiros ou mistos, com ao menos um aeródromo em território estrangeiro |
| G | Internacional Cargueira: transporte exclusivo de carga e/ou rede postal, com ao menos um aeródromo em território estrangeiro |
| X | Não informado |

O que interessa para a nossa análise são as linhas do tipo **N**

In [0]:
display(vra.groupBy(col("Código Tipo Linha")).count())

In [0]:
vra =  vra.filter(col("Código Tipo Linha") == 'N')


E agora o campo se torna obsoleto

In [0]:
vra = vra.drop("Código Tipo Linha")

In [0]:
display(vra.limit(10))

## Código DI
O Código DI (Dígito Identificador) informa o tipo de etapa de voo.

| Código DI | Descrição |
| --- | --- |
| 0 | Etapa Regular |
| 2 | Etapa Extra |
| 3 | Etapa de Retorno |
| 4 | Inclusão de Etapa |
| 6 | Etapa Não Remunerada Sem Transporte de Objetos |
| 7 | Etapa de Voo de Fretamento |
| 9 | Etapa de Voo Charter |
| D | Etapa de Voo Duplicada |
| E | Etapa Não Remunerada Com Transporte de Objetos |

In [0]:
display(vra.groupBy("Código DI").count().sort(col("count").desc()))

In [0]:
from pyspark.sql.functions import when, lit

vra = vra.withColumn(
    "descricao_etapa",
    when(col("Código DI") == "0", lit("Etapa Regular"))
    .when(col("Código DI") == "2", lit("Etapa Extra"))
    .when(col("Código DI") == "3", lit("Etapa de Retorno"))
    .when(col("Código DI") == "4", lit("Inclusão de Etapa"))
    .when(col("Código DI") == "6", lit("Etapa Não Remunerada Sem Transporte de Objetos"))
    .when(col("Código DI") == "7", lit("Etapa de Voo de Fretamento"))
    .when(col("Código DI") == "9", lit("Etapa de Voo Charter"))
    .when(col("Código DI") == "D", lit("Etapa de Voo Duplicada"))
    .when(col("Código DI") == "E", lit("Etapa Não Remunerada Com Transporte de Objetos"))
    .otherwise(lit("Não identificado"))
)

display(vra.groupBy("Código DI", "descricao_etapa").count().orderBy(col("count").desc()))

Vamos manter apenas a descricao_etapa, e não mais o código

In [0]:
vra = vra.drop("Código DI")

## Campos que virão de dimensões
Vamos eliminar da tabela vra os campos que serão provenientes de tabelas de dimensão, como informaçao de empresa e aeroporto

In [0]:
vra = vra.drop("Empresa Aérea", "Descrição Aeroporto Origem", "Descrição Aeroporto Destino")

In [0]:
display(vra.limit(10))

## Normalizar os nomes das colunas da tabela
Utilizaremos a convencao snake_case

In [0]:
import re
import unicodedata

def to_snake_case(name):
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r'[^a-zA-Z0-9]', '_', name).lower()
    name = re.sub(r'_+', '_', name).strip('_')
    return name

In [0]:
for old_name in vra.columns:
    new_name = to_snake_case(old_name)
    if new_name != old_name:
        vra = vra.withColumnRenamed(old_name, new_name)

display(vra.limit(10))

## Tipos de dados
Vamos transformar para os tipos corretos. (datas, numeros...etc)

### Tipos numéricos
Só sobrou o numero_de_asentos como numérico, numero_voo por exemplo contem caracteres alfabéticos

In [0]:
vra = vra.withColumn("numero_de_assentos", col("numero_de_assentos").cast("int"))
display(vra.select("numero_de_assentos").describe())

> vale investigar esses voos com 0 assentos

In [0]:
display(vra.filter(col("numero_de_assentos") == 0).groupBy("descricao_etapa").count())

> Um misto de tipos de voos.
> Irei manter com a seguinte premissa: Esses voos embora não estejam com passageiros, podem afetar etapas subsequentes.

### Tipos de data hora e data

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, try_to_date, coalesce, col

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dtypes = dict(vra.dtypes)
if dtypes.get("partida_prevista") == "string":
    vra = vra.withColumn("partida_prevista", to_timestamp("partida_prevista", "dd/MM/yyyy HH:mm"))
if dtypes.get("partida_real") == "string":
    vra = vra.withColumn("partida_real", to_timestamp("partida_real", "dd/MM/yyyy HH:mm"))
if dtypes.get("chegada_prevista") == "string":
    vra = vra.withColumn("chegada_prevista", to_timestamp("chegada_prevista", "dd/MM/yyyy HH:mm"))
if dtypes.get("chegada_real") == "string":
    vra = vra.withColumn("chegada_real", to_timestamp("chegada_real", "dd/MM/yyyy HH:mm"))
if dtypes.get("referencia") == "string":
    vra = vra.withColumn("referencia", coalesce(try_to_date(col("referencia"), "dd/MM/yyyy HH:mm:ss"), try_to_date(col("referencia"), "yyyy-MM-dd")))

display(vra.limit(10))

## Novas colunas
Vamos adicionar novas colunas que podem ser uteis a analise, como data de partida/chegada e tempo de atraso

### Datas de partida e chegada

In [0]:
from pyspark.sql.functions import to_date

vra = (vra
    .withColumn("data_partida", to_date("partida_prevista"))
    .withColumn("data_chegada", to_date("chegada_prevista"))
)

display(vra.limit(10))

### Tempo de atraso (partida e chegada)

In [0]:
from pyspark.sql.functions import unix_timestamp, round

vra = (vra
    .withColumn("atraso_partida_min", round((unix_timestamp("partida_real") - unix_timestamp("partida_prevista")) / 60))
    .withColumn("atraso_chegada_min", round((unix_timestamp("chegada_real") - unix_timestamp("chegada_prevista")) / 60))
)

display(vra.limit(10))

## Categorias
Uma verificacao na situacao_partida e situacao_chegada, para confirmar que estao padronizadas

In [0]:
display(vra.groupBy("situacao_partida").count())
display(vra.groupBy("situacao_chegada").count())


> As categorias já seguem uma nomenclatura padronizada

## Persistir os dados da vra no schema silver

In [0]:
vra.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("mvp_engenharia_dados.silver.silver_vra")

print(f"Tabela silver_vra criada com {vra.count():,d} linhas e {len(vra.columns)} colunas")

In [0]:
%sql
select * from mvp_engenharia_dados.silver.silver_vra limit 100

## Aeroportos

In [0]:
aeroportos = spark.table("mvp_engenharia_dados.bronze.bronze_aerodromos")

display(aeroportos.limit(10))

### Verificacao de duplicidade

In [0]:
from pyspark.sql.functions import count as count_col, col

total_linhas = aeroportos.count()

duplicatas_completas = aeroportos.groupBy(aeroportos.columns).agg(count_col("*").alias("qtd")).filter(col("qtd") > 1)
qtd_duplicatas_completas = duplicatas_completas.count()
linhas_duplicadas_completas = duplicatas_completas.agg({"qtd": "sum"}).collect()[0][0] or 0

# OACI é o identificador internacional; CIAD é o identificador civil brasileiro — ambos devem ser únicos
duplicatas_oaci = aeroportos.groupBy("Código OACI").agg(count_col("*").alias("qtd")).filter(col("qtd") > 1)
qtd_duplicatas_oaci = duplicatas_oaci.count()
linhas_duplicadas_oaci = duplicatas_oaci.agg({"qtd": "sum"}).collect()[0][0] or 0

duplicatas_ciad = aeroportos.groupBy("CIAD").agg(count_col("*").alias("qtd")).filter(col("qtd") > 1)
qtd_duplicatas_ciad = duplicatas_ciad.count()
linhas_duplicadas_ciad = duplicatas_ciad.agg({"qtd": "sum"}).collect()[0][0] or 0

print(f"Total de linhas: {total_linhas:,d}")
print(f"\n--- Duplicatas (todas as colunas idênticas) ---")
print(f"Grupos duplicados: {qtd_duplicatas_completas}")
print(f"Linhas envolvidas: {linhas_duplicadas_completas} ({linhas_duplicadas_completas/total_linhas*100:.2f}% do total)")
print(f"\n--- Duplicatas por Código OACI ---")
print(f"Códigos OACI duplicados: {qtd_duplicatas_oaci}")
print(f"Linhas envolvidas: {linhas_duplicadas_oaci} ({linhas_duplicadas_oaci/total_linhas*100:.2f}% do total)")
print(f"\n--- Duplicatas por CIAD ---")
print(f"CIADs duplicados: {qtd_duplicatas_ciad}")
print(f"Linhas envolvidas: {linhas_duplicadas_ciad} ({linhas_duplicadas_ciad/total_linhas*100:.2f}% do total)")

if qtd_duplicatas_oaci > 0:
    print("\n--- Exemplos de duplicatas por Código OACI ---")
    display(duplicatas_oaci.orderBy(col("qtd").desc()).limit(20))

> Não há duplicidade nessa tabela

In [0]:
from pyspark.sql.functions import col, count as count_col

import builtins
total_linhas = aeroportos.count()

# Vazios ('') são contados separadamente porque o CSV da ANAC pode usar string vazia em vez de NULL
resultado = []
for c in aeroportos.columns:
    nulls = aeroportos.filter(col(c).isNull()).count()
    vazios = aeroportos.filter((col(c) == "") | (col(c) == " ")).count()
    total_nulos = nulls + vazios
    pct = builtins.round(total_nulos / total_linhas * 100, 2)
    resultado.append((c, nulls, vazios, total_nulos, float(pct)))

schema = "coluna string, nulos_null int, nulos_vazio_str int, total_nulos int, pct_nulos double"
df_nulos = spark.createDataFrame(resultado, schema=schema)
display(df_nulos.orderBy(col("total_nulos").desc()))

> Temos nulos mas não temos nulos de string vazia.  
> Muitas das colunas de nulos não são relevantes a nossa analise final.

#### Nulos em Municipio

In [0]:
display(
    aeroportos.filter(col("Município").isNull()).select(
        "Código OACI",
        "CIAD",
        "Nome",
        "Município",
        "UF",
        "Município Servido",
        "UF Servido",
    )
)

> Nesses casos os nulos possuem o campo de Municipio Servido preenchidos. Acho razoavel utilizar esse campo como alternativa 

In [0]:
from pyspark.sql.functions import coalesce, col

aeroportos = (aeroportos
    .withColumn("Município", coalesce(col("Município"), col("Município Servido")))
    .withColumn("UF", coalesce(col("UF"), col("UF Servido")))
)

display(aeroportos.filter(col("Código OACI").isin("SBIZ", "SDJV", "SBME", "SBCR", "SDRS")).select("Código OACI", "Nome", "Município", "UF", "Município Servido", "UF Servido"))

#### Nulos em Municipio Servido e UF Servido

In [0]:
display(
    aeroportos.filter(col("Município Servido").isNull() | col("UF Servido").isNull()).select(
        "Código OACI", "CIAD", "Nome", "Município", "UF", "Município Servido", "UF Servido"
    )
)

> Mesma logica do tratamento anterior, só que no caminho inverso

In [0]:
aeroportos = (aeroportos
    .withColumn("Município Servido", coalesce(col("Município Servido"), col("Município")))
    .withColumn("UF Servido", coalesce(col("UF Servido"), col("UF")))
)

display(aeroportos.filter(col("Código OACI").isin("SBPW", "SSBG")).select("Código OACI", "Nome", "Município", "UF", "Município Servido", "UF Servido"))

In [0]:
import builtins
total_linhas = aeroportos.count()

resultado = []
for c in aeroportos.columns:
    nulls = aeroportos.filter(col(c).isNull()).count()
    vazios = aeroportos.filter((col(c) == "") | (col(c) == " ")).count()
    total_nulos = nulls + vazios
    pct = builtins.round(total_nulos / total_linhas * 100, 2)
    resultado.append((c, nulls, vazios, total_nulos, float(pct)))

schema = "coluna string, nulos_null int, nulos_vazio_str int, total_nulos int, pct_nulos double"
df_nulos = spark.createDataFrame(resultado, schema=schema)
display(df_nulos.orderBy(col("total_nulos").desc()))


> Como dito anteriormente, as colunas restantes com nulo não sao relevantes para a nossa analise e serão eliminadas

In [0]:
colunas_remover = ["Operação Noturna", "Operação Diurna", "Validade do Registro", "Portaria de Registro", "Link Portaria"]
aeroportos = aeroportos.drop(*colunas_remover)

print(f"Colunas removidas: {colunas_remover}")
print(f"Colunas restantes: {len(aeroportos.columns)}")
display(aeroportos.limit(5))

### Padronizacao de colunas

Vamos padronizar as escritas dos campos descritivos como Nome, Municipio etc...

In [0]:
from pyspark.sql.functions import upper

colunas_texto = ["Nome", "Município", "UF", "Município Servido", "UF Servido", "Situação"]
for c in colunas_texto:
    aeroportos = aeroportos.withColumn(c, upper(col(c)))

display(aeroportos.limit(10))

### Transformacao de Tipos

In [0]:
from pyspark.sql.functions import regexp_replace, col

# Altitude usa vírgula como separador decimal (padrão brasileiro)
aeroportos = aeroportos.withColumns({
    "LATGEOPOINT": col("LATGEOPOINT").cast("double"),
    "LONGEOPOINT": col("LONGEOPOINT").cast("double"),
    "Altitude": regexp_replace(col("Altitude"), ",", ".").cast("double"),
})

aeroportos.printSchema()
display(aeroportos.limit(10))

### Padronizacao de nomes de colunas

In [0]:
for old_name in aeroportos.columns:
    new_name = to_snake_case(old_name)
    if new_name != old_name:
        aeroportos = aeroportos.withColumnRenamed(old_name, new_name)

aeroportos.printSchema()
display(aeroportos.limit(10))

### Verificacao de valores das colunas
Como codigo_oaci e ciad foram verificados anteriormente na etapa de duplidade, não é necessario verifica novamente

In [0]:
display(
    aeroportos.groupBy("nome")
        .count()
        .orderBy(col("count").desc())
)

In [0]:
display(
    aeroportos.filter(col("nome").isin("MANOEL RIBAS", "SANTOS DUMONT", "SANTA MARIA"))
        .orderBy("nome", "uf")
)

> Os nomes duplicados são casos legitimos de aeroportos diferentes, que apenas compartilham o mesmo nome.

In [0]:
display(
    aeroportos.groupBy("municipio")
        .count()
        .orderBy(col("count").desc())
)

> Nada de notavel

In [0]:
display(
    aeroportos.groupBy("uf")
        .count()
        .orderBy(col("count").desc())
)

> Valores seguem um padrao consistente de nomenclatura dos estados

In [0]:
display(
    aeroportos.groupBy("municipio_servido")
        .count()
        .orderBy(col("count").desc())
)

In [0]:
display(
    aeroportos.groupBy("uf_servido")
        .count()
        .orderBy(col("count").desc())
)

> Novamente sem inconsistencias

In [0]:
from pyspark.sql.functions import min as min_col, max as max_col

display(
    aeroportos.agg(
        min_col("latgeopoint").alias("lat_min"),
        max_col("latgeopoint").alias("lat_max"),
        min_col("longeopoint").alias("lon_min"),
        max_col("longeopoint").alias("lon_max"),
    )
)

> Sem valores fora do range de latitude e longitude esperados

In [0]:
from pyspark.sql.functions import mean as mean_col

display(
    aeroportos.agg(
        min_col("altitude").alias("alt_min"),
        max_col("altitude").alias("alt_max"),
        mean_col("altitude").alias("alt_media"),
    )
)

> Valores de altitude dentro do esperado

In [0]:
display(
    aeroportos.groupBy("situacao")
        .count()
        .orderBy(col("count").desc())
)

> Existem aeroportos com a situacao INTERDITADO, mas devemos manter esses casos pois estamos lidando com uma base historica.

In [0]:
(aeroportos.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print(f"Tabela silver_aerodromos criada com {aeroportos.count():,d} linhas")
display(spark.table("mvp_engenharia_dados.silver.silver_aerodromos").limit(5))

### Nome de Aeroporto
Como vamos querer ver analises agrupadas por aeroportos, quero criar um nome que seja reconhecivel mas ainda unico.
Iremos fazer isso juntando o codigo_oaci (ICAO) com o nome.
Exemplo: `PARINTINS (SWPI)`

In [0]:
from pyspark.sql.functions import concat, col, lit

aeroportos = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

aeroportos = aeroportos.withColumn(
    "aerodromo",
    concat(col("nome"), lit(" ("), col("codigo_oaci"), lit(")"))
)

(aeroportos.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print(f"Tabela silver_aerodromos atualizada com {aeroportos.count():,d} linhas")
display(spark.table("mvp_engenharia_dados.silver.silver_aerodromos").limit(10))

## Empresas

In [0]:
empresas = spark.table("mvp_engenharia_dados.bronze.bronze_empresas")

display(empresas.limit(10))

### Filtro de colunas nao utilizadas

Diferentemente de outros passos anteriores, vamos primeiro ja eliminar colunas que nao serao uteis a nossa analise, ou que não tenham um possivel uso futuro.

In [0]:
colunas_remover = ["CEP", "Data", "Email", "Endereco", "Instrumento", "Telefone", "Servico"]

empresas = empresas.drop(*colunas_remover)

print(f"Colunas restantes: {empresas.columns}")
display(empresas.limit(10))

### ICAO
ICAO será nossa chave para a tabela VRA, empresas sem esse codigo não são parte do nosso escopo

In [0]:
from pyspark.sql.functions import trim

empresas = empresas.filter(
    col("ICAO").isNotNull() & (trim(col("ICAO")) != "")
)

print(f"Empresas com ICAO válido: {empresas.count():,d}")
display(empresas.limit(10))

> Isso eliminou boa parte das empresas da nossa base, o que ja era esperado. Afinal temos poucas empresas atuando em voos apenas domesticos.

### Verificacao de Nulos

In [0]:
from pyspark.sql.functions import col, count as count_col

import builtins
total_linhas = empresas.count()

# Vazios ('') são contados separadamente porque o CSV da ANAC pode usar string vazia em vez de NULL
resultado = []
for c in empresas.columns:
    nulls = empresas.filter(col(c).isNull()).count()
    vazios = empresas.filter((col(c) == "") | (col(c) == " ")).count()
    total_nulos = nulls + vazios
    pct = builtins.round(total_nulos / total_linhas * 100, 2)
    resultado.append((c, nulls, vazios, total_nulos, float(pct)))

schema = "coluna string, nulos_null int, nulos_vazio_str int, total_nulos int, pct_nulos double"
df_nulos = spark.createDataFrame(resultado, schema=schema)
display(df_nulos.orderBy(col("total_nulos").desc()))

> Com o ICAO ja filtrado, não temos nulos relevantes nessa tabela, apenas no campo `Estrangeira`, que é o código que a empresa utiliza como identificador internacional

In [0]:
from pyspark.sql.functions import when, trim, lit

empresas = empresas.withColumn(
    "Estrangeira",
    when((col("Estrangeira").isNull()) | (trim(col("Estrangeira")) == ""), lit("-"))
    .otherwise(col("Estrangeira"))
)

display(empresas.filter(col("Estrangeira") == "-").limit(10))

### Duplicidade
Vamos verificar se existe dados duplicados na tabela e de codigo ICAO

In [0]:
total_duplicadas = empresas.count() - empresas.distinct().count()

# Duplicidade por ICAO (chave de negócio)
dups_icao = (empresas
    .groupBy("ICAO")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc()))

print(f"Linhas duplicadas (todas colunas): {total_duplicadas}")
print(f"ICAO duplicados: {dups_icao.count()}")
display(dups_icao)

> Não há duplicidade

### Verificacao e padronizacao de valores

#### Coluna `Ativa`

In [0]:
display(
    empresas.groupBy("Ativa")
        .count()
        .orderBy(col("count").desc())
)

> Apenas um valor unico, nada a padronizar.

#### Vamos padronizar os campos textuais em UPPERCASE

In [0]:
from pyspark.sql.functions import upper, trim

for c in empresas.columns:
    if dict(empresas.dtypes)[c] == "string":
        empresas = empresas.withColumn(c, trim(upper(col(c))))

display(empresas.limit(10))

#### Nome empresa
Semelhante ao que fizemos com os aerportos vamos criar um campo de nome da empresa com o nome curto e o codigo ICAO


In [0]:
from pyspark.sql.functions import split, concat, lit

empresas = empresas.withColumn(
    "empresa",
    concat(split(col("Razao"), " ").getItem(0), lit(" ("), col("ICAO"), lit(")"))
)

display(empresas.select("Razao", "ICAO", "empresa"))

> Nesse resultado é possivel notar a empresa VOAR com ICAO *** o que nao é valido

In [0]:
empresas = empresas.filter(col("ICAO") != "***")

print(f"Empresas após remover ICAO='***': {empresas.count():,d}")
display(empresas.select("Razao", "ICAO", "empresa"))

### Padronizacao dos campos
Vamos padronizar as colunas como snake_case


In [0]:
for old_name in empresas.columns:
    new_name = to_snake_case(old_name)
    if new_name != old_name:
        empresas = empresas.withColumnRenamed(old_name, new_name)

display(empresas.limit(10))

### Persistir os dados no schema silver


In [0]:
(empresas.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_empresas"))

print(f"Tabela silver_empresas criada com {empresas.count():,d} linhas")
display(spark.table("mvp_engenharia_dados.silver.silver_empresas").limit(10))

## Verificação de Integridade Referencial e Refinamento

Após criar as tabelas silver, verificamos a integridade referencial entre a tabela fato (`silver_vra`) e as dimensões (`silver_empresas`, `silver_aerodromos`).

Em seguida aplicamos tratamentos complementares com dados do SIROS e filtros de qualidade para garantir 100% de cobertura no INNER JOIN.

In [0]:
from pyspark.sql.functions import col, count as count_col

vra = spark.table("mvp_engenharia_dados.silver.silver_vra")
empresas = spark.table("mvp_engenharia_dados.silver.silver_empresas")
aerodromos = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

total_vra = vra.count()

print("=" * 60)
print("VERIFICAÇÃO DE INTEGRIDADE REFERENCIAL")
print("=" * 60)
print(f"Tabela fato (silver_vra): {total_vra:,d} registros")
print(f"Dimensão empresas: {empresas.count()} registros")
print(f"Dimensão aerodromos: {aerodromos.count()} registros")

In [0]:
vra_emp = vra.join(empresas, vra.sigla_icao_empresa_aerea == empresas.icao, "left")
emp_match = vra_emp.filter(col("icao").isNotNull()).count()
emp_miss = vra_emp.filter(col("icao").isNull()).count()

vra_aero_orig = vra.join(aerodromos, vra.sigla_icao_aeroporto_origem == aerodromos.codigo_oaci, "left")
aero_orig_match = vra_aero_orig.filter(col("codigo_oaci").isNotNull()).count()
aero_orig_miss = vra_aero_orig.filter(col("codigo_oaci").isNull()).count()

vra_aero_dest = vra.join(aerodromos, vra.sigla_icao_aeroporto_destino == aerodromos.codigo_oaci, "left")
aero_dest_match = vra_aero_dest.filter(col("codigo_oaci").isNotNull()).count()
aero_dest_miss = vra_aero_dest.filter(col("codigo_oaci").isNull()).count()

print("=" * 60)
print("RESULTADO POR DIMENSÃO (LEFT JOIN)")
print("=" * 60)
print(f"\n--- Empresas ---")
print(f"  Match: {emp_match:,d} ({emp_match/total_vra*100:.2f}%)  |  Miss: {emp_miss:,d} ({emp_miss/total_vra*100:.2f}%)")
print(f"\n--- Aeródromos Origem ---")
print(f"  Match: {aero_orig_match:,d} ({aero_orig_match/total_vra*100:.2f}%)  |  Miss: {aero_orig_miss:,d} ({aero_orig_miss/total_vra*100:.2f}%)")
print(f"\n--- Aeródromos Destino ---")
print(f"  Match: {aero_dest_match:,d} ({aero_dest_match/total_vra*100:.2f}%)  |  Miss: {aero_dest_miss:,d} ({aero_dest_miss/total_vra*100:.2f}%)")

emp_keys_miss = (vra_emp.filter(col("icao").isNull())
    .groupBy("sigla_icao_empresa_aerea").agg(count_col("*").alias("registros"))
    .orderBy(col("registros").desc()))

aero_orig_keys_miss = (vra_aero_orig.filter(col("codigo_oaci").isNull())
    .groupBy("sigla_icao_aeroporto_origem").agg(count_col("*").alias("registros"))
    .orderBy(col("registros").desc()))

aero_dest_keys_miss = (vra_aero_dest.filter(col("codigo_oaci").isNull())
    .groupBy("sigla_icao_aeroporto_destino").agg(count_col("*").alias("registros"))
    .orderBy(col("registros").desc()))

print("\n--- Chaves de empresa sem match ---")
if emp_keys_miss.count() > 0:
    display(emp_keys_miss)
else:
    print("Nenhuma")

print("\n--- Chaves de aeródromo origem sem match ---")
if aero_orig_keys_miss.count() > 0:
    display(aero_orig_keys_miss)
else:
    print("Nenhuma")

print("\n--- Chaves de aeródromo destino sem match ---")
if aero_dest_keys_miss.count() > 0:
    display(aero_dest_keys_miss)
else:
    print("Nenhuma")

vra_completo = (vra
    .join(empresas, vra.sigla_icao_empresa_aerea == empresas.icao, "inner")
    .join(aerodromos, vra.sigla_icao_aeroporto_origem == aerodromos.codigo_oaci, "inner")
    .join(aerodromos.alias("d"), vra.sigla_icao_aeroporto_destino == col("d.codigo_oaci"), "inner"))

total_inner = vra_completo.count()
perda = total_vra - total_inner
print(f"\n{'=' * 60}")
print(f"INNER JOIN: {total_inner:,d} / {total_vra:,d}  (perda: {perda:,d} / {perda/total_vra*100:.2f}%)")

## Complementação com dados do SIROS/ANAC

Vamos baixar os CSVs oficiais do SIROS (aeródromos e cias) diretamente para o volume de staging e verificar se eles contêm os aeródromos e empresas que estão faltando na nossa dimensão silver.

In [0]:
import urllib.request
import os

volume_path = "/Volumes/mvp_engenharia_dados/staging/pontualidade"

urls = {
    "aerodromos_siros.csv": "https://siros.anac.gov.br/siros/registros/aerodromo/aerodromos.csv",
    "cias_siros.csv": "https://siros.anac.gov.br/siros/registros/cia/cias.csv",
}

for filename, url in urls.items():
    dest = f"{volume_path}/{filename}"
    print(f"Baixando {url} -> {dest}")
    urllib.request.urlretrieve(url, dest)
    size = os.path.getsize(dest)
    print(f"  OK: {size:,d} bytes")
    with open(dest, "r", encoding="utf-8", errors="replace") as f:
        header = f.readline().strip()
    print(f"  Header: {header}")
    print()

### Verificar aeródromos faltantes no SIROS

Ler o CSV de aeródromos baixado do SIROS e verificar se contém os códigos OACI ausentes da nossa dimensão `silver_aerodromos`.

In [0]:
from pyspark.sql.functions import upper, trim

aero_siros = (spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{volume_path}/aerodromos_siros.csv"))

print(f"Total de aeródromos no SIROS: {aero_siros.count()}")

missing_aero_orig = [r[0] for r in aero_orig_keys_miss.select("sigla_icao_aeroporto_origem").collect()]
missing_aero_dest = [r[0] for r in aero_dest_keys_miss.select("sigla_icao_aeroporto_destino").collect()]
missing_aero_all = list(set(missing_aero_orig + missing_aero_dest))

siros_codes = set([r[0] for r in aero_siros.select(upper(trim(col("SIGLA ICAO AERÓDROMO")))).distinct().collect()])

print(f"\nVerificando {len(missing_aero_all)} códigos OACI ausentes:")
print(f"{'Código':<10} {'No SIROS?':<12}")
print("-" * 25)
for code in sorted(missing_aero_all):
    in_siros = "SIM" if code in siros_codes else "NÃO"
    print(f"{code:<10} {in_siros:<12}")

found = [c for c in missing_aero_all if c in siros_codes]
not_found = [c for c in missing_aero_all if c not in siros_codes]
print(f"\nEncontrados no SIROS: {len(found)}")
print(f"Não encontrados: {len(not_found)} -> {not_found}")

### Verificar empresas faltantes no SIROS

Ler o CSV de companhias aéreas baixado do SIROS e verificar se contém os códigos ICAO ausentes da nossa dimensão `silver_empresas`.

In [0]:
cias_siros = (spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{volume_path}/cias_siros.csv"))

print(f"Total de empresas no SIROS: {cias_siros.count()}")

missing_emp = [r[0] for r in emp_keys_miss.select("sigla_icao_empresa_aerea").collect()]

siros_icao = set([r[0] for r in cias_siros.select(col("ICAO OPERADOR AÉREO")).distinct().collect() if r[0]])

print(f"\nVerificando {len(missing_emp)} códigos ICAO ausentes:")
print(f"{'Código':<10} {'No SIROS?':<12} {'Registros VRA'}")
print("-" * 40)
emp_counts = {r[0]: r[1] for r in emp_keys_miss.collect()}
for code in sorted(missing_emp):
    in_siros = "SIM" if code in siros_icao else "NÃO"
    print(f"{code:<10} {in_siros:<12} {emp_counts.get(code, 0):,d}")

found = [c for c in missing_emp if c in siros_icao]
not_found = [c for c in missing_emp if c not in siros_icao]
print(f"\nEncontrados no SIROS: {len(found)} -> {found}")
print(f"Não encontrados: {len(not_found)} -> {not_found}")
print("\nPTB = Voepass Linhas Aéreas, PAM = MAP Linhas Aéreas (identificados manualmente)")
print(f"ARN não existe no SIROS nem na Wikipedia — provável erro de digitação, apenas 1 registro")

### Inserir PTB e PAM manualmente

Como essas empresas não constam no SIROS nem na dimensão silver, mas existem registros na VRA associados a elas, vamos inseri-las manualmente.

| ICAO | Razão Social | Cidade | UF |
| --- | --- | --- | --- |
| PTB | VOEPASS LINHAS AÉREAS | RIBEIRÃO PRETO | SP |
| PAM | MAP LINHAS AÉREAS | SÃO PAULO | SP |

In [0]:
from pyspark.sql.functions import lit, concat, split

novas_empresas = spark.createDataFrame(
    [
        ("ATIVA", "RIBEIRÃO PRETO", "-", "PTB", "VOEPASS LINHAS AÉREAS", "SP", "VOEPASS (PTB)"),
        ("ATIVA", "SÃO PAULO", "-", "PAM", "MAP LINHAS AÉREAS", "SP", "MAP (PAM)"),
    ],
    schema="ativa string, cidade string, estrangeira string, icao string, razao string, uf string, empresa string"
)

empresas_atualizado = empresas.unionByName(novas_empresas)

(empresas_atualizado.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_empresas"))

print(f"silver_empresas atualizada: {empresas_atualizado.count()} empresas (antes: {empresas.count()})")
display(spark.table("mvp_engenharia_dados.silver.silver_empresas").filter(col("icao").isin("PTB", "PAM")))

## Incorporar aeródromos faltantes do SIROS

O SIROS contém todos os códigos OACI ausentes na nossa dimensão. Vamos filtrar apenas esses registros, adaptar o schema e inseri-los na dimensão.

**Mapeamento de colunas SIROS → silver_aerodromos:**

| SIROS | silver_aerodromos |
| --- | --- |
| SIGLA ICAO AERÓDROMO | codigo_oaci |
| NOME AERÓDROMO | nome |
| MUNICÍPIO AERÓDROMO | municipio / municipio_servido |
| ESTADO AERÓDROMO | uf / uf_servido |
| LATITUDE | latgeopoint |
| LONGITUDE | longeopoint |
| — | latitude, longitude (DMS), altitude, ciad = nulos |
| NOME + OACI | aerodromo (formato: `NOME (OACI)`) |

In [0]:
from pyspark.sql.functions import concat, regexp_replace

missing_codes = list(set(
    [r[0] for r in aero_orig_keys_miss.select("sigla_icao_aeroporto_origem").collect()] +
    [r[0] for r in aero_dest_keys_miss.select("sigla_icao_aeroporto_destino").collect()]
))

aeros_faltantes = aero_siros.filter(upper(trim(col("SIGLA ICAO AERÓDROMO"))).isin(missing_codes))

aeros_faltantes_fmt = (aeros_faltantes
    .select(
        upper(trim(col("SIGLA ICAO AERÓDROMO"))).alias("codigo_oaci"),
        lit(None).cast("string").alias("ciad"),
        upper(trim(col("NOME AERÓDROMO"))).alias("nome"),
        upper(trim(col("MUNICÍPIO AERÓDROMO"))).alias("municipio"),
        upper(trim(col("ESTADO AERÓDROMO"))).alias("uf"),
        upper(trim(col("MUNICÍPIO AERÓDROMO"))).alias("municipio_servido"),
        upper(trim(col("ESTADO AERÓDROMO"))).alias("uf_servido"),
        regexp_replace(col("LATITUDE"), ",", ".").cast("double").alias("latgeopoint"),
        regexp_replace(col("LONGITUDE"), ",", ".").cast("double").alias("longeopoint"),
        lit(None).cast("string").alias("latitude"),
        lit(None).cast("string").alias("longitude"),
        lit(None).cast("double").alias("altitude"),
        lit("CADASTRADO").alias("situacao"),
    )
    .withColumn("aerodromo", concat(col("nome"), lit(" ("), col("codigo_oaci"), lit(")")))
)

colunas_silver = [c.name for c in spark.table("mvp_engenharia_dados.silver.silver_aerodromos").schema.fields]
aeros_faltantes_fmt = aeros_faltantes_fmt.select(*colunas_silver)

aerodromos_atualizado = aerodromos.unionByName(aeros_faltantes_fmt)

(aerodromos_atualizado.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print(f"silver_aerodromos atualizada: {aerodromos_atualizado.count()} aeródromos (antes: {aerodromos.count()})")
display(spark.table("mvp_engenharia_dados.silver.silver_aerodromos")
    .filter(col("codigo_oaci").isin(missing_codes))
    .select("codigo_oaci", "nome", "municipio", "uf", "aerodromo"))

## Filtros de Qualidade

Após complementar as dimensões com dados do SIROS, aplicamos filtros para garantir que a análise seja estritamente doméstica e de passageiros.

### Remover voos com aeroportos estrangeiros

Como a análise é estritamente doméstica, os registros com aeroportos estrangeiros (erro de classificação na origem ANAC) serão removidos da `silver_vra`. O critério é manter apenas registros onde origem E destino são aeródromos brasileiros segundo o SIROS.

Também removemos o código ARN (1 registro), que não existe no SIROS nem na Wikipedia — provável erro de digitação.

In [0]:
aero_siros_br = aero_siros.filter(upper(trim(col("PAÍS AERÓDROMO"))) == "BRASIL")
codigos_br = set([r[0] for r in aero_siros_br.select(upper(trim(col("SIGLA ICAO AERÓDROMO")))).distinct().collect()])

vra_filtrado = vra.filter(
    col("sigla_icao_aeroporto_origem").isin(codigos_br) &
    col("sigla_icao_aeroporto_destino").isin(codigos_br)
)

removidos_estrangeiros = total_vra - vra_filtrado.count()

# Remover ARN (código inexistente, 1 registro)
vra_filtrado = vra_filtrado.filter(col("sigla_icao_empresa_aerea") != "ARN")

total_pos_filtro = vra_filtrado.count()
removidos_arn = (total_vra - removidos_estrangeiros) - total_pos_filtro

(vra_filtrado.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_vra"))

print(f"Registros originais: {total_vra:,d}")
print(f"Removidos (voos estrangeiros): {removidos_estrangeiros:,d}")
print(f"Removidos (ARN): {removidos_arn}")
print(f"silver_vra atualizada: {vra_filtrado.count():,d} registros")

### Remover aeródromos estrangeiros da dimensão

Como a análise é estritamente doméstica e os voos estrangeiros já foram filtrados da `silver_vra`, os aeródromos estrangeiros incorporados do SIROS não são mais necessários. Vamos manter apenas aeródromos com `uf` preenchido (brasileiros).

In [0]:
aerodromos_br = spark.table("mvp_engenharia_dados.silver.silver_aerodromos").filter(col("uf").isNotNull())

aerodromos_antes = spark.table("mvp_engenharia_dados.silver.silver_aerodromos").count()
removidos_aero = aerodromos_antes - aerodromos_br.count()

(aerodromos_br.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print(f"Antes: {aerodromos_antes} aeródromos")
print(f"Removidos: {removidos_aero} aeródromos estrangeiros")
print(f"Agora: {aerodromos_br.count()} aeródromos brasileiros")

### Filtrar voos de carga (numero_de_assentos = 0)

O filtro `Código Tipo Linha = N` (Doméstica Mista) aplicado anteriormente admite "transporte de passageiros **ou mistos**", permitindo que voos de carga entrassem na silver. A VRA não possui coluna de passageiros, mas `numero_de_assentos = 0` identifica aeronaves sem assentos (carga).

In [0]:
vra_sem_carga = spark.table("mvp_engenharia_dados.silver.silver_vra").filter(col("numero_de_assentos") > 0)

total_antes_carga = spark.table("mvp_engenharia_dados.silver.silver_vra").count()
total_depois_carga = vra_sem_carga.count()
removidos_carga = total_antes_carga - total_depois_carga

(vra_sem_carga.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_vra"))

print(f"Registros antes: {total_antes_carga:,d}")
print(f"Removidos (voos de carga): {removidos_carga:,d}")
print(f"silver_vra agora: {vra_sem_carga.count():,d} registros")

cargas = ["MWM", "BPC", "TOT", "TTL", "LTG"]
empresas_sem_carga = spark.table("mvp_engenharia_dados.silver.silver_empresas").filter(~col("icao").isin(cargas))

emp_antes = spark.table("mvp_engenharia_dados.silver.silver_empresas").count()
emp_depois = empresas_sem_carga.count()

(empresas_sem_carga.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_empresas"))

print(f"\nEmpresas antes: {emp_antes}")
print(f"Removidas (cargas): {emp_antes - emp_depois} ({', '.join(cargas)})")
print(f"silver_empresas agora: {emp_depois} empresas")

## Verificação Final

Re-executamos o INNER JOIN com as três dimensões atualizadas para confirmar 100% de cobertura.

In [0]:
vra_ok = spark.table("mvp_engenharia_dados.silver.silver_vra")
emp_ok = spark.table("mvp_engenharia_dados.silver.silver_empresas")
aero_ok = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

total = vra_ok.count()
inner_join = (vra_ok
    .join(emp_ok, vra_ok.sigla_icao_empresa_aerea == emp_ok.icao, "inner")
    .join(aero_ok, vra_ok.sigla_icao_aeroporto_origem == aero_ok.codigo_oaci, "inner")
    .join(aero_ok.alias("d"), vra_ok.sigla_icao_aeroporto_destino == col("d.codigo_oaci"), "inner")
).count()

perda = total - inner_join
pct = perda / total * 100

print("=" * 60)
print("VERIFICAÇÃO FINAL (após todos os refinamentos)")
print("=" * 60)
print(f"Registros VRA: {total:,d}")
print(f"Empresas:      {emp_ok.count()}")
print(f"Aeródromos:    {aero_ok.count()}")
print(f"INNER JOIN:    {inner_join:,d}")
print(f"Perda:         {perda} ({pct:.4f}%)")
print(f"Cobertura:     {inner_join / total * 100:.4f}%)")

## Investigação: UF nos aeródromos imputados do SIROS

Os 28 aeródromos incorporados do SIROS têm `ciad` nulo (campo exclusivo da ANAC, não presente no SIROS). O campo `uf` foi preenchido com `upper(trim(col("ESTADO AERÓDROMO")))` direto do CSV do SIROS. Vamos auditar se esses valores seguem o mesmo padrão dos aeródromos ANAC originais.

In [0]:
aerodromos_silver = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

siros_aero = aerodromos_silver.filter(col("ciad").isNull())
anac_aero = aerodromos_silver.filter(col("ciad").isNotNull())

print(f"Aeródromos SIROS (ciad nulo): {siros_aero.count()}")
print(f"Aeródromos ANAC originais (ciad preenchido): {anac_aero.count()}")
print()

print("=" * 80)
print("AERÓDROMOS IMPUTADOS DO SIROS — auditoria de UF")
print("=" * 80)
display(siros_aero.select(
    "codigo_oaci", "nome", "municipio", "uf", "uf_servido", "situacao"
).orderBy("uf", "codigo_oaci"))

In [0]:
ufs_siros = [r[0] for r in siros_aero.select("uf").distinct().orderBy("uf").collect() if r[0]]
ufs_anac = [r[0] for r in anac_aero.select("uf").distinct().orderBy("uf").collect() if r[0]]

print("UFs presentes nos aeródromos SIROS:")
for u in ufs_siros:
    tam = len(u) if u else 0
    flag = "← 2 letras" if tam == 2 else "← nome por extenso" if tam > 2 else "← NULL"
    print(f"  {u:<25} ({tam} chars) {flag}")

print(f"\nUFs presentes nos aeródromos ANAC:")
for u in ufs_anac:
    tam = len(u) if u else 0
    flag = "← 2 letras" if tam == 2 else "← nome por extenso" if tam > 2 else "← NULL"
    print(f"  {u:<25} ({tam} chars) {flag}")

# Verificar interseção
siros_set = set(ufs_siros)
anac_set = set(ufs_anac)
comuns = siros_set & anac_set
so_siros = siros_set - anac_set
so_anac = anac_set - siros_set

print(f"\n{'=' * 60}")
print(f"UFs em comum: {len(comuns)} -> {sorted(comuns)}")
print(f"UFs só no SIROS: {len(so_siros)} -> {sorted(so_siros)}")
print(f"UFs só na ANAC: {len(so_anac)} -> {sorted(so_anac)}")

if so_siros:
    print(f"\n INCONSISTÊNCIA: aeródromos SIROS usam padrão diferente da ANAC!")
    print(f"   SIROS usa códigos de 2 letras (SP, RJ, BA...) enquanto ANAC usa nomes por extenso (SÃO PAULO, RIO DE JANEIRO, BAHIA...)")

### Correção: Padronizar UF dos aeródromos SIROS

Os aeródromos incorporados do SIROS usam códigos de 2 letras (AM, BA, CE, MT, PA, RS) enquanto os originais da ANAC usam nomes por extenso (AMAZONAS, BAHIA, etc.). Vamos mapear para o padrão da ANAC em ambos os campos `uf` e `uf_servido`.

In [0]:
aero_corr = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

uf_map = {
    "AM": "AMAZONAS",
    "BA": "BAHIA",
    "CE": "CEARÁ",
    "MT": "MATO GROSSO",
    "PA": "PARÁ",
    "RS": "RIO GRANDE DO SUL",
}

uf_expr = col("uf")
uf_servido_expr = col("uf_servido")
for codigo, nome in uf_map.items():
    uf_expr = when(col("uf") == codigo, nome).otherwise(uf_expr)
    uf_servido_expr = when(col("uf_servido") == codigo, nome).otherwise(uf_servido_expr)

aero_corr = aero_corr.withColumn("uf", uf_expr).withColumn("uf_servido", uf_servido_expr)

(aero_corr.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print("UF dos aeródromos SIROS padronizada para nome por extenso")
print()

siros_check = spark.table("mvp_engenharia_dados.silver.silver_aerodromos").filter(col("ciad").isNull())
print("Aeródromos SIROS após correção:")
display(siros_check.select("codigo_oaci", "nome", "municipio", "uf", "uf_servido").orderBy("uf", "codigo_oaci"))

ufs_siros_novo = [r[0] for r in siros_check.select("uf").distinct().orderBy("uf").collect() if r[0]]
ufs_anac_novo = [r[0] for r in spark.table("mvp_engenharia_dados.silver.silver_aerodromos").filter(col("ciad").isNotNull()).select("uf").distinct().orderBy("uf").collect() if r[0]]
comuns = set(ufs_siros_novo) & set(ufs_anac_novo)
print(f"\nUFs em comum após correção: {len(comuns)} -> {sorted(comuns)}")

## Investigação: Faixas de atraso (situacao_partida e situacao_chegada)

As colunas `situacao_partida` e `situacao_chegada` já existem na bronze (não foram criadas neste notebook). As categorias são: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240. Vamos cruzar cada categoria com os valores reais de `atraso_partida_min` e `atraso_chegada_min` para confirmar os limites de cada faixa.

In [0]:
from pyspark.sql.functions import min as min_col, max as max_col, mean as mean_col, count as count_col, percentile_approx, col

vra_silver = spark.table("mvp_engenharia_dados.silver.silver_vra")

print("=" * 80)
print("FAIXAS DE ATRASO — PARTIDA (situacao_partida vs atraso_partida_min)")
print("=" * 80)
display(
    vra_silver.groupBy("situacao_partida").agg(
        count_col("*").alias("registros"),
        min_col("atraso_partida_min").alias("min_min"),
        max_col("atraso_partida_min").alias("max_min"),
        mean_col("atraso_partida_min").alias("media_min"),
        percentile_approx("atraso_partida_min", 0.25).alias("p25"),
        percentile_approx("atraso_partida_min", 0.50).alias("p50"),
        percentile_approx("atraso_partida_min", 0.75).alias("p75"),
    ).orderBy("min_min")
)

print("\n" + "=" * 80)
print("FAIXAS DE ATRASO — CHEGADA (situacao_chegada vs atraso_chegada_min)")
print("=" * 80)
display(
    vra_silver.groupBy("situacao_chegada").agg(
        count_col("*").alias("registros"),
        min_col("atraso_chegada_min").alias("min_min"),
        max_col("atraso_chegada_min").alias("max_min"),
        mean_col("atraso_chegada_min").alias("media_min"),
        percentile_approx("atraso_chegada_min", 0.25).alias("p25"),
        percentile_approx("atraso_chegada_min", 0.50).alias("p50"),
        percentile_approx("atraso_chegada_min", 0.75).alias("p75"),
    ).orderBy("min_min")
)

In [0]:
print("=" * 60)
print("VERIFICAÇÃO DE INCONSISTÊNCIAS")
print("=" * 60)

inconsistencias_partida = vra_silver.filter(
    ((col("situacao_partida").like("Atraso%")) & (col("atraso_partida_min") < 0)) |
    ((col("situacao_partida") == "Antecipado") & (col("atraso_partida_min") > 0)) |
    ((col("situacao_partida") == "Pontual") & (col("atraso_partida_min") > 30))
).count()

inconsistencias_chegada = vra_silver.filter(
    ((col("situacao_chegada").like("Atraso%")) & (col("atraso_chegada_min") < 0)) |
    ((col("situacao_chegada") == "Antecipado") & (col("atraso_chegada_min") > 0)) |
    ((col("situacao_chegada") == "Pontual") & (col("atraso_chegada_min") > 30))
).count()

print(f"Partida — registros com categoria incompatível com o atraso: {inconsistencias_partida:,d}")
print(f"Chegada — registros com categoria incompatível com o atraso: {inconsistencias_chegada:,d}")

if inconsistencias_partida > 0:
    print("\nExemplos de inconsistência na partida:")
    display(vra_silver.filter(
        ((col("situacao_partida").like("Atraso%")) & (col("atraso_partida_min") < 0)) |
        ((col("situacao_partida") == "Antecipado") & (col("atraso_partida_min") > 0)) |
        ((col("situacao_partida") == "Pontual") & (col("atraso_partida_min") > 30))
    ).select("situacao_partida", "atraso_partida_min", "partida_prevista", "partida_real").limit(20))

if inconsistencias_chegada > 0:
    print("\nExemplos de inconsistência na chegada:")
    display(vra_silver.filter(
        ((col("situacao_chegada").like("Atraso%")) & (col("atraso_chegada_min") < 0)) |
        ((col("situacao_chegada") == "Antecipado") & (col("atraso_chegada_min") > 0)) |
        ((col("situacao_chegada") == "Pontual") & (col("atraso_chegada_min") > 30))
    ).select("situacao_chegada", "atraso_chegada_min", "chegada_prevista", "chegada_real").limit(20))

if inconsistencias_partida == 0 and inconsistencias_chegada == 0:
    print("\n✅ Nenhuma inconsistência encontrada — as faixas estão corretas.")

### Investigação: Valores extremos de atraso

Foram identificados valores extremos: -524.906 min (365 dias) em Antecipado e 44.855 min (31 dias) em Atraso > 240. Vamos isolar esses casos para entender se são erros de data na bronze.

In [0]:
print("=" * 80)
print("REGISTROS COM ATRASO EXTREMO (|atraso| > 1 dia = 1440 min)")
print("=" * 80)

extremos_partida = vra_silver.filter(
    (col("atraso_partida_min") > 1440) | (col("atraso_partida_min") < -1440)
)
extremos_chegada = vra_silver.filter(
    (col("atraso_chegada_min") > 1440) | (col("atraso_chegada_min") < -1440)
)

print(f"Partida: {extremos_partida.count()} registros")
print(f"Chegada: {extremos_chegada.count()} registros")
print()

if extremos_partida.count() > 0:
    print("--- EXTREMOS NA PARTIDA ---")
    display(extremos_partida.select(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "sigla_icao_aeroporto_origem",
        "sigla_icao_aeroporto_destino",
        "partida_prevista",
        "partida_real",
        "atraso_partida_min",
        "situacao_partida",
        "referencia",
    ).orderBy(col("atraso_partida_min").asc()))

if extremos_chegada.count() > 0:
    print("\n--- EXTREMOS NA CHEGADA ---")
    display(extremos_chegada.select(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "sigla_icao_aeroporto_origem",
        "sigla_icao_aeroporto_destino",
        "chegada_prevista",
        "chegada_real",
        "atraso_chegada_min",
        "situacao_chegada",
        "referencia",
    ).orderBy(col("atraso_chegada_min").asc()))

### Remoção: Registros com atraso extremo (erro de data)

Removemos registros onde `|atraso_partida_min| > 1440` ou `|atraso_chegada_min| > 1440` (mais de 1 dia de diferença). Esses 210 registros (0,008%) são erros de digitação de data na fonte ANAC — não representam atrasos reais e distorcem as estatísticas.

In [0]:
vra_filtrado = spark.table("mvp_engenharia_dados.silver.silver_vra")

total_antes = vra_filtrado.count()

vra_filtrado = vra_filtrado.filter(
    (col("atraso_partida_min").isNull() | (col("atraso_partida_min") >= -1440) & (col("atraso_partida_min") <= 1440)) &
    (col("atraso_chegada_min").isNull() | (col("atraso_chegada_min") >= -1440) & (col("atraso_chegada_min") <= 1440))
)

total_depois = vra_filtrado.count()
removidos = total_antes - total_depois

print(f"Registros antes:  {total_antes:,d}")
print(f"Removidos:        {removidos:,d} ({removidos/total_antes*100:.4f}%)")
print(f"Registros agora:  {total_depois:,d}")

(vra_filtrado.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_vra"))

print("\nsilver_vra atualizada — registros com atraso extremo removidos")

print("\n--- Novos limites de atraso ---")
display(spark.table("mvp_engenharia_dados.silver.silver_vra").agg(
    min_col("atraso_partida_min").alias("partida_min"),
    max_col("atraso_partida_min").alias("partida_max"),
    min_col("atraso_chegada_min").alias("chegada_min"),
    max_col("atraso_chegada_min").alias("chegada_max"),
))

In [0]:
vra_atual = spark.table("mvp_engenharia_dados.silver.silver_vra")
total = vra_atual.count()

print(f"Total de registros: {total:,d}")
print()
print(f"{'Faixa |atraso|':<25} {'Partida':>12} {'%':>8} {'Chegada':>12} {'%':>8}")
print("-" * 70)

faixas = [
    ("0-30 min", 0, 30),
    ("31-60 min", 31, 60),
    ("61-120 min", 61, 120),
    ("121-240 min", 121, 240),
    ("241-480 min (8h)", 241, 480),
    ("481-720 min (12h)", 481, 720),
    ("721-1440 min (24h)", 721, 1440),
]

for nome, lo, hi in faixas:
    p = vra_atual.filter((col("atraso_partida_min") >= lo) & (col("atraso_partida_min") <= hi)).count()
    c = vra_atual.filter((col("atraso_chegada_min") >= lo) & (col("atraso_chegada_min") <= hi)).count()
    print(f"{nome:<25} {p:>12,d} {p/total*100:>7.4f}% {c:>12,d} {c/total*100:>7.4f}%")

print()
for nome, lo, hi in faixas:
    p = vra_atual.filter((col("atraso_partida_min") <= -lo) & (col("atraso_partida_min") >= -hi)).count()
    c = vra_atual.filter((col("atraso_chegada_min") <= -lo) & (col("atraso_chegada_min") >= -hi)).count()
    print(f"{'-' + nome:<25} {p:>12,d} {p/total*100:>7.4f}% {c:>12,d} {c/total*100:>7.4f}%")

print("\n" + "=" * 70)
print("DISTRIBUIÇÃO ACUMULADA — |atraso| > threshold")
print("=" * 70)
print(f"{'Threshold':<25} {'Partida':>12} {'%':>8} {'Chegada':>12} {'%':>8}")
print("-" * 70)

for nome, thr in [("|atraso| > 60 min", 60), ("|atraso| > 120 min", 120), ("|atraso| > 240 min", 240), ("|atraso| > 480 min", 480), ("|atraso| > 720 min", 720), ("|atraso| > 1440 min", 1440)]:
    p = vra_atual.filter((col("atraso_partida_min") > thr) | (col("atraso_partida_min") < -thr)).count()
    c = vra_atual.filter((col("atraso_chegada_min") > thr) | (col("atraso_chegada_min") < -thr)).count()
    print(f"{nome:<25} {p:>12,d} {p/total*100:>7.4f}% {c:>12,d} {c/total*100:>7.4f}%")

### Hipotese: erro de 1 dia na data real

Se os atrasos de ~±1440 min sao causados por erro de digitacao de 1 dia na data real (nao no horario), entao deslocar a data real em ±1 dia deve produzir atrasos plasiveis (ex: -720 a +720 min = 12h). Vamos testar essa hipotese.

In [0]:
from pyspark.sql.functions import when as when_fn, expr, round as round_fn, unix_timestamp

vra_silver = spark.table("mvp_engenharia_dados.silver.silver_vra")

suspeitos_p = vra_silver.filter(
    ((col("atraso_partida_min") >= -1440) & (col("atraso_partida_min") <= -720)) |
    ((col("atraso_partida_min") >= 720) & (col("atraso_partida_min") <= 1440))
)

suspeitos_p = suspeitos_p.withColumn(
    "partida_real_ajustada",
    when_fn(col("atraso_partida_min") < 0, expr("partida_real + INTERVAL 1 day"))
    .otherwise(expr("partida_real - INTERVAL 1 day"))
).withColumn(
    "atraso_corrigido_min",
    round_fn((unix_timestamp("partida_real_ajustada") - unix_timestamp("partida_prevista")) / 60)
)

total_p = suspeitos_p.count()
print(f"Registros suspeitos (partida): {total_p}")

plausiveis_p = suspeitos_p.filter(
    (col("atraso_corrigido_min") >= -720) & (col("atraso_corrigido_min") <= 720)
).count()
print(f"Plasiveis apos deslocar 1 dia: {plausiveis_p} de {total_p} ({plausiveis_p/total_p*100:.1f}%)")
print()
print("--- AMOSTRA (partida) ---")
display(suspeitos_p.select(
    "sigla_icao_empresa_aerea", "numero_voo",
    "partida_prevista", "partida_real", "partida_real_ajustada",
    "atraso_partida_min", "atraso_corrigido_min",
    "situacao_partida"
).orderBy("atraso_partida_min").limit(30))

suspeitos_c = vra_silver.filter(
    ((col("atraso_chegada_min") >= -1440) & (col("atraso_chegada_min") <= -720)) |
    ((col("atraso_chegada_min") >= 720) & (col("atraso_chegada_min") <= 1440))
)

suspeitos_c = suspeitos_c.withColumn(
    "chegada_real_ajustada",
    when_fn(col("atraso_chegada_min") < 0, expr("chegada_real + INTERVAL 1 day"))
    .otherwise(expr("chegada_real - INTERVAL 1 day"))
).withColumn(
    "atraso_chegada_corrigido",
    round_fn((unix_timestamp("chegada_real_ajustada") - unix_timestamp("chegada_prevista")) / 60)
)

total_c = suspeitos_c.count()
plausiveis_c = suspeitos_c.filter(
    (col("atraso_chegada_corrigido") >= -720) & (col("atraso_chegada_corrigido") <= 720)
).count()
print(f"\nRegistros suspeitos (chegada): {total_c}")
print(f"Plasiveis apos deslocamento: {plausiveis_c} de {total_c} ({plausiveis_c/total_c*100:.1f}%)")

print(f"\n{'=' * 60}")
print("CONCLUSAO DA HIPOTESE")
print(f"{'=' * 60}")
if plausiveis_p/total_p > 0.9 and plausiveis_c/total_c > 0.9:
    print(f">90% dos casos ficam plasiveis ao deslocar 1 dia.")
    print("HIPOTESE CONFIRMADA: os atrasos de ~1 dia sao erros de data, nao de horario.")
    print("Correcao: deslocar a data real em +1 dia (antecipacao) ou -1 dia (atraso).")
else:
    print(f"Apenas {plausiveis_p/total_p*100:.1f}% (partida) e {plausiveis_c/total_c*100:.1f}% (chegada) ficam plasiveis.")
    print("HIPOTESE REJEITADA: o deslocamento de 1 dia nao explica a maioria dos casos.")

### Correção: Deslocamento de 1 dia nos registros com atraso de 12h-24h

Registros com |atraso| entre 720 e 1440 min (12h-24h) sao quase certamente erros de data na fonte ANAC — o horario esta correto, mas a data foi registrada no dia anterior ou posterior. Aplicamos o deslocamento de ±1 dia na data real e recalculamos os campos derivados.

**Campos afetados:**
- `partida_real` / `chegada_real` — deslocados ±1 dia (antecipacao: +1 dia, atraso: -1 dia)
- `atraso_partida_min` / `atraso_chegada_min` — recalculados a partir das novas datas reais
- `situacao_partida` / `situacao_chegada` — reclassificados conforme os novos atrasos (apenas nos registros deslocados)

**Campos nao afetados:**
- `partida_prevista` / `chegada_prevista` — nao mudam
- `data_partida` / `data_chegada` — extraidas de `partida_prevista` / `chegada_prevista`, nao mudam
- `referencia` — mes de publicacao ANAC, nao muda
- Demais colunas (empresa, voo, equipamento, aerodromos, assentos, descricao_etapa) — nao mudam

In [0]:
from pyspark.sql.functions import when, expr, round as round_fn, unix_timestamp, min as min_col, max as max_col

vra = spark.table("mvp_engenharia_dados.silver.silver_vra")
total_antes = vra.count()

vra = vra.withColumn(
    "_shifted_p",
    when(
        ((col("atraso_partida_min") >= -1440) & (col("atraso_partida_min") <= -720)) |
        ((col("atraso_partida_min") >= 720) & (col("atraso_partida_min") <= 1440)),
        True
    ).otherwise(False)
).withColumn(
    "_shifted_c",
    when(
        ((col("atraso_chegada_min") >= -1440) & (col("atraso_chegada_min") <= -720)) |
        ((col("atraso_chegada_min") >= 720) & (col("atraso_chegada_min") <= 1440)),
        True
    ).otherwise(False)
)

shifted_p = vra.filter(col("_shifted_p")).count()
shifted_c = vra.filter(col("_shifted_c")).count()

vra = vra.withColumn(
    "partida_real",
    when(
        (col("_shifted_p") & (col("atraso_partida_min") < 0)),
        expr("partida_real + INTERVAL 1 day")
    ).when(
        (col("_shifted_p") & (col("atraso_partida_min") > 0)),
        expr("partida_real - INTERVAL 1 day")
    ).otherwise(col("partida_real"))
).withColumn(
    "chegada_real",
    when(
        (col("_shifted_c") & (col("atraso_chegada_min") < 0)),
        expr("chegada_real + INTERVAL 1 day")
    ).when(
        (col("_shifted_c") & (col("atraso_chegada_min") > 0)),
        expr("chegada_real - INTERVAL 1 day")
    ).otherwise(col("chegada_real"))
)

vra = vra.withColumn(
    "atraso_partida_min",
    round_fn((unix_timestamp("partida_real") - unix_timestamp("partida_prevista")) / 60)
).withColumn(
    "atraso_chegada_min",
    round_fn((unix_timestamp("chegada_real") - unix_timestamp("chegada_prevista")) / 60)
)

vra = vra.withColumn(
    "situacao_partida",
    when(col("_shifted_p"),
        when(col("atraso_partida_min") < 0, "Antecipado")
        .when(col("atraso_partida_min") <= 30, "Pontual")
        .when(col("atraso_partida_min") <= 60, "Atraso 30-60")
        .when(col("atraso_partida_min") <= 120, "Atraso 60-120")
        .when(col("atraso_partida_min") <= 240, "Atraso 120-240")
        .otherwise("Atraso > 240")
    ).otherwise(col("situacao_partida"))
).withColumn(
    "situacao_chegada",
    when(col("_shifted_c"),
        when(col("atraso_chegada_min") < 0, "Antecipado")
        .when(col("atraso_chegada_min") <= 30, "Pontual")
        .when(col("atraso_chegada_min") <= 60, "Atraso 30-60")
        .when(col("atraso_chegada_min") <= 120, "Atraso 60-120")
        .when(col("atraso_chegada_min") <= 240, "Atraso 120-240")
        .otherwise("Atraso > 240")
    ).otherwise(col("situacao_chegada"))
)

vra = vra.drop("_shifted_p", "_shifted_c")

print(f"Registros: {total_antes:,d}")
print(f"Partidas deslocadas: {shifted_p:,d}")
print(f"Chegadas deslocadas: {shifted_c:,d}")

(vra.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_vra"))

print("silver_vra atualizada")
print()
display(spark.table("mvp_engenharia_dados.silver.silver_vra").agg(
    min_col("atraso_partida_min").alias("partida_min"),
    max_col("atraso_partida_min").alias("partida_max"),
    min_col("atraso_chegada_min").alias("chegada_min"),
    max_col("atraso_chegada_min").alias("chegada_max"),
))

### Comparativo Bronze vs Silver

Reproduzimos cada filtro aplicado na transformacao bronze → silver para quantificar quantos registros foram removidos em cada etapa, separados por categoria:

**Qualidade de dados:** duplicatas, nulos em campos obrigatorios, atrasos extremos (erro de data na fonte ANAC)

**Regra de negocio/premissas:** filtro de voos domesticos (Tipo Linha = N), voos de carga (assentos = 0), aerodromos estrangeiros/ARN

In [0]:
vra = spark.table("mvp_engenharia_dados.bronze.bronze_vra")

bronze_total = vra.count()

c1 = vra.count()
vra = vra.dropDuplicates()
c2 = vra.count()
dup_removidas = c1 - c2

c1 = vra.count()
vra = vra.dropna(subset=["Código Tipo Linha"])
c2 = vra.count()
nulo_tipolinha = c1 - c2

c1 = vra.count()
vra = vra.dropna(subset=["Partida Prevista", "Chegada Prevista"], how='any')
c2 = vra.count()
nulo_prevista = c1 - c2

c1 = vra.count()
vra = vra.dropna(subset=["Partida Real", "Chegada Real"], how='any')
c2 = vra.count()
nulo_real = c1 - c2

apos_qualidade = vra.count()

c1 = vra.count()
vra = vra.filter(col("Código Tipo Linha") == 'N')
c2 = vra.count()
domestico_removido = c1 - c2

c1 = vra.count()
vra = vra.filter(col("Número de Assentos") > 0)
c2 = vra.count()
carga_removida = c1 - c2

silver_atual = spark.table("mvp_engenharia_dados.silver.silver_vra")
apos_regras = vra.count()
extremos_removidos = 133
foreign_arn = apos_regras - silver_atual.count() - extremos_removidos

silver_final = silver_atual.count()

qualidade_total = dup_removidas + nulo_tipolinha + nulo_prevista + nulo_real + extremos_removidos
regra_total = domestico_removido + carga_removida + foreign_arn

print(f"BRONZE (origem):           {bronze_total:>12,d}")
print(f"SILVER (final):            {silver_final:>12,d}")
print(f"Perda total:               {bronze_total - silver_final:>12,d} ({(bronze_total - silver_final)/bronze_total*100:.2f}%)")
print()
print("=" * 60)
print("QUALIDADE DE DADOS")
print("=" * 60)
print(f"Duplicatas:                 {dup_removidas:>10,d}")
print(f"Nulos - Codigo Tipo Linha:  {nulo_tipolinha:>10,d}")
print(f"Nulos - Datas previstas:     {nulo_prevista:>10,d}")
print(f"Nulos - Datas reais:         {nulo_real:>10,d}")
print(f"Atrasos extremos (>1440min):{extremos_removidos:>10,d}")
print(f"  Subtotal:                 {qualidade_total:>10,d}")
print()
print("=" * 60)
print("REGRA DE NEGOCIO / PREMISSAS")
print("=" * 60)
print(f"Nao domesticos (I/C/G/X):   {domestico_removido:>10,d}")
print(f"Voos de carga (assentos=0): {carga_removida:>10,d}")
print(f"Estrangeiros/ARN:           {foreign_arn:>10,d}")
print(f"  Subtotal:                 {regra_total:>10,d}")

## Documentação das Tabelas Silver

Após os refinamentos, documentamos todas as tabelas e colunas do schema `mvp_engenharia_dados.silver` com comentários no catálogo Unity Catalog. Os comentários ficam visíveis no Data Explorer e facilitam o consumo pelas camadas downstream.

In [0]:
%sql

COMMENT ON TABLE mvp_engenharia_dados.silver.silver_vra IS 'Tabela contendo os registros de voos domésticos de passageiros da VRA — Voo Regular Ativo da ANAC. Filtro original: Código Tipo Linha = N (Doméstica Mista). Filtro adicional: numero_de_assentos > 0 (remoção de voos de carga). Cada registro representa uma etapa de voo realizada por uma companhia aérea entre dois aeródromos brasileiros. Inclui horários previstos e realizados, atrasos calculados em minutos e situação de pontualidade classificada em faixas.';

COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.sigla_icao_empresa_aerea IS 'Código ICAO (designador de 3 letras) da empresa aérea operadora do voo. Será chave estrangeira para silver_empresas.icao no schema gold';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.numero_voo IS 'Número do voo atribuído pela empresa aérea, concatenado com o código ICAO forma o identificador único do voo';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.modelo_equipamento IS 'Modelo da aeronave utilizada no voo no padrão ICAO (ex: B738, A320, E190)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.numero_de_assentos IS 'Número total de assentos da aeronave utilizada no voo. Voos com assentos = 0 (carga) foram filtrados';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.sigla_icao_aeroporto_origem IS 'Código OACI (4 letras) do aeródromo de origem do voo. Será chave estrangeira para silver_aerodromos.codigo_oaci no schema gold';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.partida_prevista IS 'Data e hora previstas para a decolagem do voo (horário de Brasília)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.partida_real IS 'Data e hora reais da decolagem do voo (horário de Brasília)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.sigla_icao_aeroporto_destino IS 'Código OACI (4 letras) do aeródromo de destino do voo. Será chave estrangeira para silver_aerodromos.codigo_oaci no schema gold';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.chegada_prevista IS 'Data e hora previstas para o pouso do voo (horário de Brasília)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.chegada_real IS 'Data e hora reais do pouso do voo (horário de Brasília)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.referencia IS 'Mês de referência da publicação: período ao qual as decolagens previstas ou realizadas pertencem';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.situacao_partida IS 'Situação do voo na partida classificada em faixas: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240 (minutos)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.situacao_chegada IS 'Situação do voo na chegada classificada em faixas: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240 (minutos)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.descricao_etapa IS 'Descrição do tipo de etapa do voo conforme ANAC: Etapa de Voo Regular, Etapa de Voo Charter, Etapa de Voo Extra, etc.';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.data_partida IS 'Data (sem hora) da partida prevista do voo, extraída de partida_prevista';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.data_chegada IS 'Data (sem hora) da chegada prevista do voo, extraída de chegada_prevista';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.atraso_partida_min IS 'Atraso na partida em minutos, calculado como partida_real - partida_prevista. Valores negativos indicam antecipação';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_vra.atraso_chegada_min IS 'Atraso na chegada em minutos, calculado como chegada_real - chegada_prevista. Valores negativos indicam antecipação';

In [0]:
%sql

COMMENT ON TABLE mvp_engenharia_dados.silver.silver_empresas IS 'Tabela de empresas aéreas (companhias aéreas) operando no Brasil. Fonte: aerodromos/cias da ANAC, complementada manualmente com PTB (Voepass) e PAM (MAP) que não constavam no SIROS. Empresas exclusivamente de carga (MWM, BPC, TOT, TTL, LTG) foram removidas após filtragem de voos sem assentos. Cada empresa é identificada unicamente pelo código ICAO.';

COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_empresas.ativa IS 'Situação cadastral da empresa: ATIVA, INATIVA, etc.';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_empresas.cidade IS 'Cidade da sede da empresa aérea';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_empresas.estrangeira IS 'Código designador internacional da empresa aérea (ex: WD, 2S, TT). Valor "-" quando a empresa não possui designador internacional';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_empresas.icao IS 'Código ICAO (designador de 3 letras) da empresa aérea — identificador único da tabela. Será chave primária no schema gold, com junção a silver_vra.sigla_icao_empresa_aerea';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_empresas.razao IS 'Razão social da empresa aérea';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_empresas.uf IS 'Estado (UF) da sede da empresa aérea';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_empresas.empresa IS 'Nome curto da empresa no formato PRIMEIRA_PALAVRA_RAZAO (ICAO), ex: GOL (GLO), TOTAL (TTL)';

In [0]:
%sql

COMMENT ON TABLE mvp_engenharia_dados.silver.silver_aerodromos IS 'Tabela de aeródromos brasileiros. Fonte: dados da ANAC, complementada com 8 aeródromos do SIROS que estavam ausentes e eram referenciados pela VRA (20 aeródromos estrangeiros do SIROS foram removidos). O campo UF foi padronizado para nome por extenso (AMAZONAS, BAHIA, etc.) em ambas as fontes. Cada aeródromo é identificado pelo código OACI.';

COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.codigo_oaci IS 'Código de 4 letras utilizado internacionalmente para identificação do aeródromo, conforme regras da OACI. No Brasil, os códigos iniciam com SB, SD, SI, SJ, SN, SS e SW. Identificador único da tabela; será chave primária no schema gold';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.ciad IS 'Código de Identificação do Aeródromo: identificador único de aeródromos civis, formado por dois caracteres da UF e quatro dígitos sequenciais (formato UF0000). Nulo para aeródromos incorporados do SIROS';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.nome IS 'Nome do aeródromo';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.municipio IS 'Município onde o aeródromo está localizado';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.uf IS 'Estado (UF) onde o aeródromo está localizado, padronizado em nome por extenso (AMAZONAS, BAHIA, SÃO PAULO, etc.). Nulo para aeródromos estrangeiros (removidos desta dimensão)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.municipio_servido IS 'Município servido pelo aeródromo (pode diferir do município de localização quando o aeródromo atende cidades vizinhas)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.uf_servido IS 'Estado (UF) servido pelo aeródromo';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.latgeopoint IS 'Latitude do ponto de referência do aeródromo em graus decimais (sistema de referência WGS-84)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.longeopoint IS 'Longitude do ponto de referência do aeródromo em graus decimais (sistema de referência WGS-84)';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.latitude IS 'Latitude do ponto de referência do aeródromo em graus, minutos e segundos (sistema de referência WGS-84). Nulo para aeródromos incorporados do SIROS';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.longitude IS 'Longitude do ponto de referência do aeródromo em graus, minutos e segundos (sistema de referência WGS-84). Nulo para aeródromos incorporados do SIROS';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.altitude IS 'Altitude do ponto de referência mais elevado na área de pouso, em metros. Nulo para aeródromos incorporados do SIROS';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.situacao IS 'Situação cadastral do aeródromo: CADASTRADO, INTERDITADO';
COMMENT ON COLUMN mvp_engenharia_dados.silver.silver_aerodromos.aerodromo IS 'Nome do aeródromo com código OACI no formato NOME (OACI), ex: GUARULHOS (SBGR)';

In [0]:
%sql

SELECT
  table_name,
  column_name AS coluna,
  data_type AS tipo,
  comment AS comentario
FROM mvp_engenharia_dados.information_schema.columns
WHERE table_catalog = 'mvp_engenharia_dados'
  AND table_schema = 'silver'
  AND table_name IN ('silver_vra', 'silver_empresas', 'silver_aerodromos')
ORDER BY table_name, column_name;

## Resumo do Notebook

Este notebook transformou as tabelas da camada **bronze** em tabelas tratadas na camada **silver**, aplicando limpeza, padronizacao, filtros de qualidade e tipagem adequada. Foram processadas tres tabelas:

### 1. `silver_vra` (tabela fato - voos)
- **Origem:** `mvp_engenharia_dados.bronze.bronze_vra`
- **Resultado:** 2.732.331 linhas e 18 colunas
- **Tratamentos aplicados:**
  - Remocao de duplicatas
  - Eliminacao de voos cancelados e sem datas previstas/reais (perda de ~7%)
  - Filtro apenas para linhas domesticas de passageiros (`Codigo Tipo Linha` = 'N')
  - Remocao de colunas nao relevantes: `Justificativa`, `Codeshare`, `Situacao Voo`, `Codigo Tipo Linha`
  - Criacao de `descricao_etapa` a partir do mapeamento do `Codigo DI`
  - Remocao de campos descritivos que virao de tabelas dimensao (`Empresa Aerea`, `Descricao Aeroporto Origem/Destino`)
  - Conversao de tipos: `numero_de_assentos` para inteiro, datas para `timestamp`/`date` (timezone America/Sao_Paulo)
  - Criacao de colunas derivadas: `data_partida`, `data_chegada`, `atraso_partida_min`, `atraso_chegada_min`
  - Padronizacao snake_case em todos os nomes de coluna
  - Verificacao das categorias de `situacao_partida` e `situacao_chegada` (ja padronizadas)
  - Remocao de 133 registros com atraso extremo (|atraso| > 1440 min = 1 dia) — erros de data na fonte ANAC
  - Deslocamento de ±1 dia em 660 partidas e 665 chegadas com atraso de 12h-24h — erros de data corrigidos preservando o horario

### 2. `silver_aerodromos` (dimensao - aeroportos)
- **Origem:** `mvp_engenharia_dados.bronze.bronze_aerodromos`
- **Resultado:** 504 linhas e 15 colunas
- **Tratamentos aplicados:**
  - Verificacao de duplicidade (nenhuma encontrada por OACI, CIAD ou linha completa)
  - Tratamento de nulos: `Municipio`/`UF` preenchidos com `Municipio Servido`/`UF Servido` e vice-versa via `coalesce`
  - Remocao de colunas administrativas com nulos: `Operacao Noturna`, `Operacao Diurna`, `Validade do Registro`, `Portaria de Registro`, `Link Portaria`
  - Padronizacao UPPERCASE em campos textuais
  - Conversao de tipos: `latgeopoint`/`longeopoint` para double, `altitude` para double (correcao de virgula decimal)
  - Padronizacao snake_case nos nomes de coluna
  - Criacao da coluna `aerodromo` no formato `NOME (CODIGO_OACI)`, ex: `PARINTINS (SWPI)`
  - Complementacao com 8 aerodromos do SIROS que estavam ausentes e eram referenciados pela VRA
  - Remocao de 20 aerodromos estrangeiros do SIROS (analise estritamente domestica)
  - Padronizacao de UF dos aerodromos SIROS de 2 letras para nome por extenso

### 3. `silver_empresas` (dimensao - empresas aereas)
- **Origem:** `mvp_engenharia_dados.bronze.bronze_empresas`
- **Resultado:** 16 linhas e 7 colunas
- **Tratamentos aplicados:**
  - Remocao de colunas nao utilizadas: `CEP`, `Data`, `Email`, `Endereco`, `Instrumento`, `Telefone`, `Servico`
  - Filtro apenas para empresas com codigo `ICAO` valido (nao nulo e nao vazio)
  - Preenchimento do unico valor vazio em `Estrangeira` com `-`
  - Verificacao de duplicidade (nenhuma encontrada)
  - Verificacao da coluna `Ativa` (todas ATIVA)
  - Padronizacao UPPERCASE em todos os campos textuais
  - Criacao da coluna `empresa` no formato `PRIMEIRO_NOME (ICAO)`, ex: `GOL (GLO)`
  - Remocao da empresa com ICAO `***` (invalido)
  - Insercao manual de PTB (Voepass) e PAM (MAP) nao constavam no SIROS
  - Remocao de 5 empresas exclusivamente de carga (MWM, BPC, TOT, TTL, LTG)
  - Padronizacao snake_case nos nomes de coluna

### Padroes aplicados em todas as tabelas
- Nomes de coluna em **snake_case** (funcao `to_snake_case` definida na celula 51)
- Campos textuais em **UPPERCASE** com `trim`
- Tipos de dados adequados (int, double, timestamp, date)
- Persistencia no schema `mvp_engenharia_dados.silver` com `mode("overwrite")` e `overwriteSchema`

### 4. Verificação de Integridade Referencial e Refinamento

Após criar as tabelas silver, verificamos a integridade referencial e aplicamos tratamentos complementares:

- **Estado inicial:** 2.736.859 registros VRA, 19 empresas, 496 aeródromos, perda de 1,75% no INNER JOIN
- **Complementação de empresas (PTB e PAM):** Baixamos CSV do SIROS/ANAC, identificamos PTB = Voepass e PAM = MAP (não constavam no SIROS), inserimos manualmente. Recuperados ~38.778 registros.
- **Complementação de aeródromos (SIROS):** Baixamos CSV de aeródromos do SIROS (10.560 mundiais), encontramos os 28 códigos OACI ausentes, incorporamos na dimensão. Recuperados ~11.840 registros.
- **Remoção de voos estrangeiros:** 44 voos com aeroportos fora do Brasil (erro de classificação ANAC) + 1 registro com empresa ARN inexistente.
- **Limpeza da dimensão de aeródromos:** Removidos 20 aeródromos estrangeiros incorporados do SIROS que não eram mais referenciados.
- **Filtro de voos de carga:** Removidos 4.350 voos com `numero_de_assentos = 0` + 5 empresas exclusivamente de carga (MWM, BPC, TOT, TTL, LTG).
- **Remoção de atrasos extremos:** 133 registros com |atraso| > 1440 min (mais de 1 dia) removidos — erros de digitação de data na fonte ANAC.
- **Correção de atrasos 12h-24h:** Deslocamento de ±1 dia em 660 partidas e 665 chegadas — o horário estava correto, apenas a data foi registrada no dia errado. Atraso, situação e datas reais recalculados.
- **Padronização de UF SIROS:** Aeródromos incorporados do SIROS usavam códigos de 2 letras (SP, RJ, BA...), corrigidos para nome por extenso (SÃO PAULO, RIO DE JANEIRO, BAHIA...).
- **Documentação:** `COMMENT ON TABLE` e `COMMENT ON COLUMN` nas três tabelas via Unity Catalog.

**Estado final:** 2.732.331 registros VRA, 16 empresas, 504 aeródromos, **100% de cobertura no INNER JOIN**.

**Bronze vs Silver:** 3.565.162 (bronze) → 2.732.331 (silver). Perda de 832.831 registros (23,36%), sendo 250.377 por qualidade de dados e 582.454 por regras de negócio/premissas.
